# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pyrootutils
from pathlib import Path

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Parameters

In [ ]:
N = 10
MAX_RECORDINGS = 500
CLIPS_PER_SPECIES = 2500
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

ACTIVE_COLLECTIONS = ["diff_species"] # ["diff_family", "diff_genus", "diff_species"]

# Select species

In [ ]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS,dl_more = True)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species:   0%|          | 0/11167 [00:00<?, ?it/s]

Species pool: 1805 species with 100 XC recordings


In [5]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
Phylloscopus         32                 1                  1                 1
      Turdus         29                 1                  1                 1
   Setophaga         23                 1                  1                 1
    Emberiza         17                 1                  1                 1
       Vireo         15                 1                  1                 1
      Anthus         15                 1                  1                 1
Thamnophilus         14                 1                  1                 1
     Curruca         14                 1                  1                 1
Acrocephalus         12                 1                  1                 1
    Calidris         11                 1                  1                 1
       Strix         11                 1                  1                 1
      Trogon   

In [6]:
TARGET_GENUS  = "Phylloscopus"
TARGET_FAMILY = "Tyrannidae (Tyrant Flycatchers)"
TARGET_ORDER  = "Passeriformes"

In [ ]:
collections: dict[str, list[str]] = {}

if "diff_species" in ACTIVE_COLLECTIONS:
    diff_species = select_same_genus(TARGET_GENUS, N, pool)
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    diff_genus = select_same_family(TARGET_FAMILY, N, pool)
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    diff_family = select_same_order(TARGET_ORDER, N, pool)
    collections["diff_family"] = [s.scientific_name for s in diff_family]

collections_to_use = {k: collections[k] for k in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_species:
  Phylloscopus collybita
  Phylloscopus trochilus
  Phylloscopus sibilatrix
  Phylloscopus inornatus
  Phylloscopus humei
  Phylloscopus ibericus
  Phylloscopus trochiloides
  Phylloscopus fuscatus
  Phylloscopus bonelli
  Phylloscopus xanthoschistos

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_species']


# Download from Xeno-canto

In [8]:
from building.download import download_species_list

all_species = list({name for names in collections.values() for name in names})
print(f"Downloading {len(all_species)} unique species ({MAX_RECORDINGS} recordings each)…")

downloaded = download_species_list(all_species, max_recordings=MAX_RECORDINGS)

print("\nDownload summary:")
for name, paths in downloaded.items():
    print(f"  {name}: {len(paths)} files")

[Phylloscopus trochilus] existing_total=0 processed=1000 existing_xc=0 target=500
[Phylloscopus trochilus] already complete, download=0
  Phylloscopus trochilus: 0 files
[Phylloscopus sibilatrix] existing_total=0 processed=1000 existing_xc=0 target=500
[Phylloscopus sibilatrix] already complete, download=0
  Phylloscopus sibilatrix: 0 files
[Phylloscopus inornatus] existing_total=0 processed=800 existing_xc=0 target=500
[Phylloscopus inornatus] already complete, download=0
  Phylloscopus inornatus: 0 files
[Phylloscopus ibericus] existing_total=0 processed=734 existing_xc=0 target=500
[Phylloscopus ibericus] already complete, download=0
  Phylloscopus ibericus: 0 files
[Phylloscopus collybita] existing_total=0 processed=1208 existing_xc=0 target=500
[Phylloscopus collybita] already complete, download=0
  Phylloscopus collybita: 0 files
[Phylloscopus trochiloides] existing_total=0 processed=602 existing_xc=0 target=500
[Phylloscopus trochiloides] already complete, download=0
  Phyllosco

[Phylloscopus fuscatus] downloaded_new=0 total_now=0 missing_after=500 skipped_duplicate=454 skipped_processed=454 skipped_quality=0 failed=0
  Phylloscopus fuscatus: 0 files
[Phylloscopus humei] existing_total=0 processed=661 existing_xc=0 target=500
[Phylloscopus humei] already complete, download=0
  Phylloscopus humei: 0 files
[Phylloscopus xanthoschistos] existing_total=0 processed=727 existing_xc=0 target=500
[Phylloscopus xanthoschistos] already complete, download=0
  Phylloscopus xanthoschistos: 0 files
[Phylloscopus bonelli] existing_total=0 processed=659 existing_xc=0 target=500
[Phylloscopus bonelli] already complete, download=0
  Phylloscopus bonelli: 0 files

Download summary:
  Phylloscopus trochilus: 0 files
  Phylloscopus sibilatrix: 0 files
  Phylloscopus inornatus: 0 files
  Phylloscopus ibericus: 0 files
  Phylloscopus collybita: 0 files
  Phylloscopus trochiloides: 0 files
  Phylloscopus fuscatus: 0 files
  Phylloscopus humei: 0 files
  Phylloscopus xanthoschistos: 0

# BirdNET validation + dataset assembly

In [11]:
from building.dataset import validate_and_build_all

MAX_PER_CLASS = 1750

dataset_paths = validate_and_build_all(
    collections_to_use,
    clips_per_species=CLIPS_PER_SPECIES,
    max_class_size_training=MAX_PER_CLASS,
    threshold=BIRDNET_THRESHOLD,
)

print("\nDatasets ready:")
for name, path in dataset_paths.items():
    print(f"  {name}: {path}")  


BirdNET subsamples: diff_species (10 species, threshold=0.92)...


diff_species: 100%|██████████| 10/10 [00:00<00:00, 33.81it/s]



Building dataset: diff_species
  → /home/nathan/Documents/multi-chirp/datasets/diff_species

Datasets ready:
  diff_species: /home/nathan/Documents/multi-chirp/datasets/diff_species


# Sanity check

In [10]:
from pathlib import Path

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips")
        for species, count in sorted(species_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {species:20s} {count:5d}")


diff_species
  training     31793 clips
    Phylloscopus_sibilatrix  3837
    Phylloscopus_xanthoschistos  3649
    Phylloscopus_collybita  3589
    Phylloscopus_ibericus  3575
    Phylloscopus_trochilus  3571
    Phylloscopus_bonelli  3469
    Phylloscopus_inornatus  3278
    Phylloscopus_trochiloides  2672
    Phylloscopus_humei    2404
    Phylloscopus_fuscatus  1749
  validation   11052 clips
    Phylloscopus_sibilatrix  1828
    Phylloscopus_xanthoschistos  1392
    Phylloscopus_collybita  1305
    Phylloscopus_ibericus  1293
    Phylloscopus_trochilus  1272
    Phylloscopus_bonelli  1131
    Phylloscopus_inornatus   994
    Phylloscopus_trochiloides   757
    Phylloscopus_humei     706
    Phylloscopus_fuscatus   374
  testing      10648 clips
    Phylloscopus_sibilatrix  1837
    Phylloscopus_xanthoschistos  1394
    Phylloscopus_collybita  1309
    Phylloscopus_ibericus  1297
    Phylloscopus_trochilus  1276
    Phylloscopus_bonelli  1132
    Phylloscopus_inornatus  1001
    P